# Lesson 02 — Chunking

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/safar-byte/rag-from-scratch-to-production/blob/main/lessons/02-chunking/colab.ipynb)

*Nothing downstream can recover information that chunking destroyed.*

**What you will learn**

- The core tension: small chunks retrieve precisely, large chunks preserve context
- Fixed-size vs recursive splitting — implemented by hand, then compared on the corpus
- Overlap: what it is for and what it costs
- Characters vs tokens, and why the limit you set is not the limit the model sees
- Two silent chunking bugs (orphaned headings, mid-word overlap) and how to see them
- A first *measured* comparison of chunk sizes — a preview of lesson 03

| | |
|---|---|
| ⏱️ Time | 40–50 min |
| 🖥️ Runtime | CPU is fine |
| 🤖 Needs the generator (Ollama) | no — retrieval only |

**Before this:** [01-naive-rag](https://colab.research.google.com/github/safar-byte/rag-from-scratch-to-production/blob/main/lessons/01-naive-rag/colab.ipynb)

> **How to use this notebook.** Run the cells top to bottom (`Shift+Enter`). Every
> time you see **🔮 Predict**, stop and write down what you expect *before* running
> the next cell — the gap between prediction and result is where the learning is.
> Answers to the check-your-understanding questions are hidden behind ▶ toggles.

## 0. Setup

This cell clones the course repo into the Colab runtime, installs the
dependencies, and (if `USE_LLM` is ticked) installs and starts
[Ollama](https://ollama.com) with the small `qwen2.5:1.5b` generator. It is safe
to re-run. Everything in the repo's `ragkit/` library becomes importable.

If an import fails after the install, use **Runtime → Restart session** and run
this cell again.

In [ ]:
# @title ⚙️ Setup — run me first
USE_LLM = False  # @param {type:"boolean"}
BRANCH = "main"  # @param {type:"string"}

import subprocess
import sys
from pathlib import Path


def _find_repo():
    for p in [Path.cwd(), *Path.cwd().parents]:
        if (p / "ragkit").is_dir() and (p / "colab" / "colab_setup.py").exists():
            return p
    return None


REPO = _find_repo()
if REPO is None:
    REPO = Path("/content/rag-from-scratch-to-production")
    if not REPO.exists():
        subprocess.run(
            [
                "git",
                "clone",
                "--depth",
                "1",
                "--branch",
                BRANCH,
                "https://github.com/safar-byte/rag-from-scratch-to-production.git",
                str(REPO),
            ],
            check=True,
        )
sys.path.insert(0, str(REPO / "colab"))
from colab_setup import compare_runs, eval_frames, results_frame, setup  # noqa: F401

LLM_READY = setup(llm=USE_LLM)

## 1. Why chunking is the highest-leverage decision

A fact split across two chunks so that neither one states it is **gone**. The best
reranker in the world cannot reorder its way to a passage that does not exist, and the
best model cannot ground an answer in text it was never shown.

And chunking is the part everyone copies from a tutorial without measuring.

### The core tension

| | Small chunks | Large chunks |
|---|---|---|
| Embedding | sharp — about one thing | a blurry average of many topics |
| Retrieval precision | high | low — "strongly retrieved by nothing" |
| Context for the answer | fragmented — a fact may land half in each chunk | preserved |
| Prompt cost per hit | low | high |

"512 tokens with 64 overlap" is a common starting point. It is a starting point, not a
recommendation. Lesson 08 shows a way to refuse the trade-off (search small, answer
with big).

## 2. Two chunkers, by hand

We will use one short passage so you can *see* every boundary.

In [ ]:
sample = """# Dimensionality

Embedding models produce vectors of a fixed size. BGE-small produces 384 dimensions.
Larger models produce 1024 or more, which costs memory and search time.

## Normalisation

Vectors are normalised to unit length. After normalisation, cosine similarity equals the
dot product, which is cheaper to compute. Most vector databases exploit this."""


def show(chunks):
    for i, c in enumerate(chunks):
        print(f"──── chunk {i} ({len(c)} chars) " + "─" * 30)
        print(c)
    print()

### Fixed-size: cut every N characters

The simplest thing that works, and the baseline everything else is measured against.
`step = size − overlap`, so each chunk repeats the last `overlap` characters of the
previous one.

In [ ]:
def fixed_chunks(text, size=120, overlap=20):
    step = size - overlap
    return [text[i : i + size] for i in range(0, len(text), step) if text[i : i + size].strip()]


show(fixed_chunks(sample))

🔍 Look at where the cuts fall: mid-word, mid-sentence, and the `## Normalisation`
heading is separated from its section. Every one of those is a small loss of
information.

### Recursive: split on the most meaningful boundary that fits

Try separators in order — paragraph break, line break, sentence end, comma, space — and
only fall through to a cruder one when a piece is still too big. This is the idea behind
`ragkit.ingest.RecursiveChunker` (and LangChain's splitter of the same name).

In [ ]:
SEPARATORS = ["\n\n", "\n", ". ", " ", ""]


def recursive_chunks(text, size=120, separators=SEPARATORS):
    if len(text) <= size:
        return [text] if text.strip() else []
    sep, rest = separators[0], separators[1:]
    if sep == "":  # last resort: hard cut
        return [text[i : i + size] for i in range(0, len(text), size)]
    pieces, buffer = [], ""
    for part in text.split(sep):
        candidate = f"{buffer}{sep}{part}" if buffer else part
        if len(candidate) <= size:
            buffer = candidate  # keep growing this chunk
            continue
        if buffer:
            pieces.append(buffer)
        if len(part) > size:  # still too big → finer separator
            pieces.extend(recursive_chunks(part, size, rest))
            buffer = ""
        else:
            buffer = part
    if buffer:
        pieces.append(buffer)
    return [p for p in pieces if p.strip()]


show(recursive_chunks(sample, size=200))

Much better: cuts land on paragraph and sentence boundaries. But look closely at the
first chunk: it *ends* with `## Normalisation`. The **heading has been stranded at the
end of the previous chunk**, away from the section it introduces. That is bug #1 below.
(Try `size=120` too: the heading becomes a 16-character chunk of its own — a variant of
the same problem.)

## 3. The library's chunkers on the real corpus

`ragkit` has `FixedSizeChunker` and `RecursiveChunker`. Both record `start_char` /
`end_char` into the source document — the bookkeeping that later makes citations and
parent-document retrieval possible. The lesson script prints a comparison table:

In [ ]:
!{sys.executable} lessons/02-chunking/run.py

**How to read it.** "Split mid-sentence" is a crude damage proxy: a chunk that does not
end in `.`, `!`, `?` or `:` was almost certainly cut through a sentence. Fixed-size
cutting breaks ~4 sentences in 5; recursive splitting at the same size gets that to ~1
in 20.

⚠️ **This table does not tell you which is best.** It measures *damage*, not *answer
quality*. Notice how tempting it is to conclude "recursive(512/64) wins" from a column
that never touches retrieval. Section 7 actually measures it.

Let's look at the size distributions too.

In [ ]:
import matplotlib.pyplot as plt

from ragkit.ingest import FixedSizeChunker, RecursiveChunker, chunk_documents, load_documents

documents = load_documents()
configs = [
    FixedSizeChunker(512, 64),
    RecursiveChunker(256, 64),
    RecursiveChunker(512, 64),
    RecursiveChunker(1024, 64),
]

fig, axes = plt.subplots(1, len(configs), figsize=(16, 3), sharey=True)
for ax, chunker in zip(axes, configs, strict=True):
    sizes = [len(c.text) for c in chunk_documents(documents, chunker)]
    ax.hist(sizes, bins=20, color="#4C72B0")
    ax.set_title(f"{chunker.name}\n{len(sizes)} chunks")
    ax.set_xlabel("characters")
axes[0].set_ylabel("count")
plt.tight_layout()
plt.show()

Fixed-size chunks are nearly all the same length (a spike at the limit). Recursive
chunks spread out *below* the limit, because they stop at a natural boundary instead of
the exact character count.

### Offsets: every chunk knows where it came from

In [ ]:
chunks = chunk_documents(documents, RecursiveChunker(512, 64))
by_id = {d.doc_id: d for d in documents}
c = chunks[10]
print(c.chunk_id, "→ chars", c.start_char, "to", c.end_char, "of", c.doc_id)
print("\nchunk text      :", repr(c.text[:90]))
print("source[start:]  :", repr(by_id[c.doc_id].text[c.start_char : c.start_char + 90]))

🔍 **Look closely — the two lines do not quite match.** The chunk starts with its
*overlap* (the tail of the previous chunk), which was joined with a single space where
the source has a blank line, so `start_char` is off by a character and `inferred.
## Overlap` reads slightly differently from the source. The offsets are approximate
whenever overlap is on. Nothing raises; a citation highlighter would just be one
character off. This was found while writing this notebook and is left as an exercise
(see the end) — a nice example of how small and quiet real chunking bugs are.

## 4. Characters vs tokens

`ragkit`'s chunk sizes are in **characters**. Models count **tokens** (sub-word pieces).
For English prose a token is roughly 4 characters, but identifiers, numbers and code
tokenise much worse. If a downstream limit is in tokens — an embedding model's max
sequence length, a prompt budget — measure in tokens.

In [ ]:
import tiktoken

enc = tiktoken.get_encoding("cl100k_base")
for text in [
    "Reranking reorders a shortlist of candidate chunks.",
    "TX-4491 MBV2-2846 FUSION_CONSTANT=60 port 8788",
    chunks[10].text,
]:
    n_tokens = len(enc.encode(text))
    print(
        f"{len(text):5d} chars → {n_tokens:4d} tokens  ({len(text) / n_tokens:.1f} chars/token)"
        f"  {text[:50]!r}"
    )

BGE-small truncates input at **512 tokens**. A 512-*character* chunk is ~120 tokens, so
nothing is lost here — but a 4,000-character chunk would be silently truncated by the
embedder, and the end of every chunk would simply not be searchable. Another bug that
never raises.

## 5. Overlap

Overlap repeats the tail of one chunk at the head of the next, so a fact sitting on a
boundary survives intact in at least one chunk. Typically 10–20% of the chunk size.

Costs: more chunks to store and embed, and **near-duplicate results** — two adjacent
chunks can both match the same query and waste two prompt slots on the same sentence.

In [ ]:
pairs = [(a, b) for a, b in zip(chunks, chunks[1:], strict=False) if a.doc_id == b.doc_id]
a, b = pairs[3]
print("END of chunk", a.chunk_id, ":\n   …", a.text[-120:])
print("\nSTART of chunk", b.chunk_id, ":\n  ", b.text[:120], "…")

## 6. Two bugs that never raise an error

Both were found while building this lesson; `tests/test_chunker.py` now guards each.

### Bug 1 — orphaned headings

A greedy buffer fills up to the size limit and flushes. A heading that happens to land
at the boundary therefore ends the *previous* chunk. Both chunks get worse: the section
body loses its most discriminative keyword ("Dimensionality" is exactly what someone
searches for), and the previous chunk gains a label for content it does not contain.

The fix, `_merge_headings`, glues each heading to the section it introduces *before* the
buffer sees it. Let's switch it off and count the damage.

In [ ]:
import ragkit.ingest.chunker as chunker_module


def chunks_ending_in_heading(chunk_list):
    return [c for c in chunk_list if c.text.rstrip().splitlines()[-1].lstrip().startswith("#")]


fixed = chunk_documents(documents, RecursiveChunker(512, 64))

original = chunker_module._merge_headings
chunker_module._merge_headings = lambda parts, separator: parts  # break it
try:
    broken = chunk_documents(documents, RecursiveChunker(512, 64))
finally:
    chunker_module._merge_headings = original  # always restore

print("chunks ending in an orphaned heading — with the fix:", len(chunks_ending_in_heading(fixed)))
print("chunks ending in an orphaned heading — without it:  ", len(chunks_ending_in_heading(broken)))
for c in chunks_ending_in_heading(broken)[:3]:
    print(f"\n{c.chunk_id} ends with:\n   …{c.text[-90:]!r}")

### Bug 2 — mid-word overlap

Taking the last N *characters* of the previous chunk produces overlaps that begin
`"rly always retrieval depth"` instead of `"nearly always"`. The fragment `rly` is
noise to the embedder and a junk token to keyword search. `_tail_words` snaps the tail
forward to a word boundary.

In [ ]:
from ragkit.ingest.chunker import _tail_words

previous = "Shortlist depth is nearly always retrieval depth times three in practice"
print("naive  last 50 chars :", repr(previous[-50:]))
print("_tail_words(…, 50)   :", repr(_tail_words(previous, 50)))

In [ ]:
!{sys.executable} -m pytest tests/test_chunker.py -q

## 7. Measure it: which chunk size retrieves best?

This is a preview of lesson 03's evaluation harness. It asks 44 test questions, each
labelled with the document(s) that answer it, and reports **recall@k** — the fraction
of the right documents found in the top *k* results.

🔮 **Predict before running:** rank these four configurations by recall@1 —
`fixed(512/64)`, `recursive(256/64)`, `recursive(512/64)`, `recursive(1024/64)`.
Write your ranking down.

Each configuration gets its own throw-away index under `/tmp` so the course's main
index is untouched.

In [ ]:
from pathlib import Path

from ragkit.config import Settings
from ragkit.eval.run import run_eval
from ragkit.pipeline import RagPipeline
from ragkit.providers import get_embedder

embedder = get_embedder()  # load once, share across all four indexes
sweep = {}
for chunker in [
    FixedSizeChunker(512, 64),
    RecursiveChunker(256, 64),
    RecursiveChunker(512, 64),
    RecursiveChunker(1024, 64),
]:
    settings = Settings(chroma_path=Path(f"/tmp/l02/{chunker.name}/.chroma"))
    pipe = RagPipeline(settings, strategy="dense")
    pipe._embedder = embedder
    info = pipe.ingest(chunker=chunker)
    sweep[f"{chunker.name} · {info['chunks']} chunks"] = run_eval(
        settings, pipeline=pipe, retrieval_only=True, quiet=True
    )

compare_runs(sweep).drop(columns=["ms/query"]).round(3)

**Compare with your prediction.** The differences between adjacent rows are often one
or two questions out of 44 — so treat small gaps as noise, not findings. What you
*can* see:

- Whether the damage column from section 3 (fixed-size breaks 80% of sentences)
  translates into worse retrieval — it often matters less than you'd guess, because
  embeddings are robust to a ragged edge.
- Whether bigger chunks help or hurt on *this* corpus. Short, focused documents
  tolerate large chunks; a corpus of long multi-topic documents would not.

And notice how many configurations score near 1.0 at recall@5. That is the ceiling
effect again — the corpus is too small for k=5 to discriminate, which is why lesson 03
reports k=1 and k=3 too.

## 8. Strategies not implemented here

- **Semantic chunking** embeds each sentence and breaks where consecutive similarity
  drops, so boundaries land where the topic turns. Costs one embedding per sentence at
  ingest.
- **Structural chunking** uses the document's own markup (Markdown headings, HTML
  sections, PDF layout). Where real structure exists it usually beats anything
  inferred — `_merge_headings` is a small step in that direction.

## 9. Check your understanding

**1. Why is a 2000-token chunk spanning twelve topics "strongly retrieved by nothing"?**

<details><summary>▶ Show answer</summary>

Its single embedding is roughly an average of all twelve topics, so it is
moderately similar to many queries and highly similar to none. A small, focused
chunk about the query's topic will out-score it every time.

</details>

**2. Why should a golden test set label relevant *documents*, not relevant *chunks*?**

<details><summary>▶ Show answer</summary>

Chunk ids change every time you change `chunk_size` or the chunker. A chunk-level
golden set would silently invalidate itself during exactly the experiment you most
want to run — the chunk-size sweep you just did.

</details>

**3. Overlap makes boundary facts survive. Name two costs.**

<details><summary>▶ Show answer</summary>

More chunks (storage, embedding time) and near-duplicate results that waste prompt
slots — which is why deduplication before prompt assembly matters.

</details>

## ✍️ Exercises

1. **Find the damage.** Pick a fact from `data/03-hybrid-search.md` (e.g. "k is
   conventionally set to 60"). Under `FixedSizeChunker(256, 0)`, is it ever split so
   that no single chunk states it? Use the cell below.
2. **Overlap 0 vs 64.** Re-run the section 7 sweep with `RecursiveChunker(512, 0)`.
   Does overlap move recall at all on this corpus?
3. **Fix the offset bug** from section 3 in `RecursiveChunker.split` so that
   `document.text[start_char:end_char]` reproduces the chunk exactly, and add a test
   to `tests/test_chunker.py` that would have caught it.
4. **Think ahead.** Lesson 07 prepends an LLM-written context sentence to each chunk.
   Would that let you use *smaller* chunks without losing context?

In [ ]:
fact = "conventionally"
tiny = chunk_documents([by_id["03-hybrid-search.md"]], FixedSizeChunker(256, 0))
for c in tiny:
    if fact in c.text:
        print(c.chunk_id, "…", c.text[max(0, c.text.find(fact) - 120) : c.text.find(fact) + 60])

## Key takeaways

1. Chunking is lossy and nothing downstream can undo it.
2. Recursive splitting preserves sentence and paragraph boundaries far better than fixed-size cutting.
3. Overlap protects boundary facts at the cost of duplicates.
4. Limits are in characters here; models count tokens — check you are not silently truncating.
5. A damage statistic is not a quality measurement. Measure retrieval, and distrust one-question differences.

## Next

**[Lesson 03 — The evaluation harness ⭐](https://colab.research.google.com/github/safar-byte/rag-from-scratch-to-production/blob/main/lessons/03-evaluation-harness/colab.ipynb)**